# 145 고정 RG3 미성형 모델을 비라벨 RH에 적용
학습 정상566/미성형8, A/B/C/D 고정구조. 판정선은 학습Run 안쪽 예측으로 계산. 비라벨은 추론만. S14 RH중 모든알려진라벨Timestamp제외, 충돌/이력부족/다른운전모드 플래그. 정답없으므로성능지표불가. 모델학습시 sparse std10과 추론dense std10차이를명시.

In [ ]:
from pathlib import Path
import os,sys,json,time
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd();t=root/'reports/tables';outdir=root/'reports';data_root=root/'data'
na={};nb={};nd={}
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.cluster import KMeans\nmemo={};audit=[]\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for view in ['std24']:\n  X=views[view];imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  full=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy)\n  fullscore=expit(full.decision_function(b))\n  pos=np.flatnonzero(yy==1);neg=np.flatnonzero(normal)\n  k=min(2,len(np.unique(a[pos],axis=0)))\n  modes=KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(a[pos]) if k>1 else np.zeros(len(pos),int)\n  experts=[]\n  for mode in np.unique(modes):\n   chosen=np.r_[neg,pos[modes==mode]]\n   scores=[]\n   for C in [.1,1.]:\n    model=SVC(C=C,class_weight='balanced',gamma='scale').fit(a[chosen],yy[chosen])\n    scores.append(expit(model.decision_function(b)))\n   for C in [.1,1.]:\n    model=LogisticRegression(C=C,class_weight='balanced',max_iter=3000).fit(a[chosen],yy[chosen])\n    scores.append(model.predict_proba(b)[:,1])\n   experts.append(np.mean(scores,axis=0))\n  expert=np.max(experts,axis=0)\n  out[view+'_mode_max']=expert\n  out[view+'_full']=fullscore\n  out[view+'_geometric']=np.sqrt(expert*fullscore)\n  out[view+'_agreement_min']=np.minimum(expert,fullscore)\n  out[view+'_union_max']=np.maximum(expert,fullscore)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",na)
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\nfrom sklearn.neighbors import NearestNeighbors\nmemo={};audit=[]\ncontext=lab[[v+'_raw' for v in ['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']]].to_numpy(float)\ndef localize(ref,qq,X):\n normal=ref[y[ref]==0]\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[normal])\n xx=imp.transform(X)\n ci=SimpleImputer(strategy='median').fit(context[normal])\n cx=ci.transform(context);cs=RobustScaler().fit(cx[normal]);cx=cs.transform(cx)\n nn=NearestNeighbors(n_neighbors=min(31,len(normal))).fit(cx[normal])\n near=nn.kneighbors(cx[qq],return_distance=False)\n floor=np.maximum(xx[normal].std(axis=0)*.2,1e-6)\n out=[]\n for i,ids in zip(qq,near):\n  peers=normal[ids];peers=peers[peers!=i][:30]\n  out.append((xx[i]-np.median(xx[peers],axis=0))/np.maximum(xx[peers].std(axis=0),floor))\n return np.asarray(out)\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for tag,X in [('localphysical12',views['physical12'])]:\n  a=localize(ref,ref,X);b=localize(ref,q,X)\n  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  for name,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\n   pp=[]\n   for seed in [42,7]:\n    m=cls(n_estimators=120,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\n    pp.append(m.predict_proba(b)[:,1])\n   out[tag+'_'+name]=np.mean(pp,axis=0)\n  for C in [.1,1.]:\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\n   out[tag+'_svm'+str(C)]=expit(m.decision_function(b))\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\n  out[tag+'_lr']=m.predict_proba(b)[:,1]\n  out[tag+'_mean']=np.mean([out[tag+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n",nb)
exec("from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\nval=lambda n:dense[n].to_numpy(float)\nvs=lambda n:val(n+'__past10_std')\nvd=lambda n:val(n+'__delta1')\nve=lambda n:val(n+'__dev10')\nthermal={\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\n}\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\n# Collapse several opposite-direction disturbances into shared intensity/count features.\njointbase=np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Hopper_Temperature'),vs('Max_Injection_Speed'),vs('Cushion_Position'),vs('Plasticizing_Time'),vs('Barrel_Temperature_5'),vs('Average_Back_Pressure')])\nmemo={}\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n yy=y[ref];normal=yy==0\n assert set(g[ref]).isdisjoint(g[q])\n imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(jointbase[ref][normal])\n a=imp.transform(jointbase[ref]);b=imp.transform(jointbase[q]);sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n def features(x):\n  j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))\n  return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])\n model=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(features(a),yy)\n memo[key]=model.predict_proba(features(b))[:,1]\n return memo[key]\n",nd)
sourceC="from pathlib import Path\nimport os,sys,time\nimport numpy as np,pandas as pd\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import SVC\nfrom sklearn.metrics import confusion_matrix,average_precision_score,precision_recall_curve\nfrom scipy.special import expit\nroot=Path.cwd()\nif not (root/'reports').exists():root=root.parent\nt=root/'reports/tables'\ntd=root/'data/taein'\nsys.dont_write_bytecode=True;sys.path.insert(0,str(td))\nold=Path.cwd()\ntry:\n os.chdir(td);import semi_normal as SN;D=SN.load_std('rg3')\nfinally:os.chdir(old)\nlab=D['lab'].copy()\nlab=lab[lab['eval']].reset_index(drop=True)\nr=lab; g=lab.run_id.to_numpy();y=lab.type.eq('미성형').to_numpy(int)\nuse=lab.type.isin(['정상','미성형']).to_numpy()\nstd=pd.DataFrame({n:r[n.replace('|std10','')+'_raw'].groupby(r.run_id).transform(lambda s:s.rolling(10,min_periods=3).std()) for n in D['names']})\n# Current inclusive std preserved to compare TaEin's formulation. No label used in rolling values.\nraw=lambda v:r[v+'_raw']\ndelta=lambda v:raw(v).groupby(r.run_id).diff()\ndev=lambda v:raw(v)-raw(v).groupby(r.run_id).transform(lambda s:s.shift(1).rolling(10,min_periods=3).median())\nratio=lambda a,b:a/b.where(b.abs()>1e-6)\nphys=pd.DataFrame({\n 'peak_minus_switch':raw('Max_Injection_Pressure')-raw('Max_Switch_Over_Pressure'),\n 'switch_fraction':ratio(raw('Max_Switch_Over_Pressure'),raw('Max_Injection_Pressure')),\n 'fill_fraction':ratio(raw('Filling_Time'),raw('Injection_Time')),\n 'fill_per_speed':ratio(raw('Filling_Time'),raw('Max_Injection_Speed')),\n 'meter_cushion_gap':raw('Plasticizing_Position')-raw('Cushion_Position'),\n 'meter_cushion_delta_gap':delta('Plasticizing_Position')-delta('Cushion_Position'),\n 'recovery_fraction':ratio(raw('Plasticizing_Time'),raw('Cycle_Time')),\n 'recovery_dev':dev('Plasticizing_Time'),\n 'backpressure_dev':dev('Average_Back_Pressure'),\n 'thermal_gap':raw('Barrel_Temperature_5')-(raw('Mold_Temperature_3')+raw('Mold_Temperature_4'))/2,\n 'pressure_dev':dev('Max_Injection_Pressure'),\n 'cushion_dev':dev('Cushion_Position')})\ncompact=['Filling_Time','Plasticizing_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Pressure','Max_Switch_Over_Pressure','Barrel_Temperature_5','Hopper_Temperature']\nviews={'std24':std,'compact8':std[[v+'|std10' for v in compact]],'physical12':phys}\nviews={n:x.replace([np.inf,-np.inf],np.nan).to_numpy(float) for n,x in views.items()}\n# Dense sensor history uses actual earlier raw shots; no future recovery features.\ndense=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])\ndense=dense[dense['product'].eq('RG3')].set_index('TimeStamp').loc[lab.TimeStamp].reset_index(drop=True)\nval=lambda n:dense[n].to_numpy(float)\nvs=lambda n:val(n+'__past10_std')\nvd=lambda n:val(n+'__delta1')\nve=lambda n:val(n+'__dev10')\nthermal={\n 'shock6':np.column_stack([ve('Hopper_Temperature'),ve('Barrel_Temperature_4'),vd('Barrel_Temperature_5'),vd('Barrel_Temperature_3'),vd('Mold_Temperature_4'),vs('Cushion_Position')]),\n 'contrast6':np.column_stack([np.log1p(vs('Barrel_Temperature_1')),np.log1p(vs('Barrel_Temperature_5')),np.log1p(vs('Hopper_Temperature')),np.log((vs('Barrel_Temperature_5')+.01)/(vs('Hopper_Temperature')+.01)),np.log1p(vs('Average_Back_Pressure')),np.log1p(vs('Plasticizing_Time'))]),\n 'cooling6':np.column_stack([ve('Mold_Temperature_3'),ve('Mold_Temperature_4'),ve('Barrel_Temperature_4'),vd('Mold_Temperature_4'),vs('Barrel_Temperature_6'),vd('Barrel_Temperature_3')])\n}\nthermal={k:np.where(np.isfinite(v),v,np.nan) for k,v in thermal.items()}\nmemo={};audit=[]\ndef predict(ref,q):\n key=(tuple(ref),tuple(q))\n if key in memo:return memo[key]\n assert set(g[ref]).isdisjoint(g[q])\n yy=y[ref];normal=yy==0;out={}\n for view,X in {'shock6':thermal['shock6']}.items():\n  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(X[ref][normal])\n  a=imp.transform(X[ref]);b=imp.transform(X[q]);sc=RobustScaler().fit(a[normal])\n  a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)\n  for tag,cls in [('rf',RandomForestClassifier),('et',ExtraTreesClassifier)]:\n   pp=[]\n   for seed in [42,7]:\n    m=cls(n_estimators=100,max_depth=4,min_samples_leaf=2,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)\n    pp.append(m.predict_proba(b)[:,1])\n   out[view+'_'+tag]=np.mean(pp,axis=0)\n  for C in [.1,1.]:\n   m=SVC(C=C,gamma='scale',class_weight='balanced').fit(a,yy)\n   out[view+'_svm'+str(C)]=expit(m.decision_function(b))\n  m=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)\n  out[view+'_lr']=m.predict_proba(b)[:,1]\n  out[view+'_mean']=np.mean([out[view+'_'+n] for n in ['rf','et','svm0.1','svm1.0','lr']],axis=0)\n memo[key]=out;audit.append(dict(ref_rows=len(ref),ref_shortshot=int(yy.sum()),q_rows=len(q),run_overlap=0,unlabeled_rows=0))\n return out\ndef threshold(yy,s,limit=None):\n p,r,th=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12)\n opts=[]\n for j,c in enumerate(th):\n  fp=int(((s>=c)&(yy==0)).sum())\n  if limit is None or fp<=int(np.floor((yy==0).sum()*limit)):opts.append((f[j],r[j],-fp,c))\n return max(opts)[-1] if opts else float(np.max(s)+1e-6)\n"
nc=[]
for seeds in [[42,7],[11,23],[101,37]]:
 ns={};exec(sourceC.replace('for seed in [42,7]:','for seed in '+str(seeds)+':'),ns);nc.append(ns)
lab=na['lab'];nlab=len(lab);yn=na['y'];gn=na['g'];usable=na['use']
assert nlab==591 and int(yn[usable].sum())==8 and int(usable.sum())==574
sensor_names=[x.replace('|std10','') for x in na['D']['names']]
u_all=pd.read_csv(data_root/'original_split/moldset_unlabeled_rg3.csv',parse_dates=['TimeStamp'])
target_rows=u_all.loc[u_all.EQUIP_CD.eq('S14')&u_all.PART_NAME.str.contains('RH',na=False)].copy()
target=target_rows.drop_duplicates('TimeStamp').sort_values('TimeStamp').reset_index(drop=True)
target_raw_count=len(target_rows)
records=[]
known_timestamps=set(pd.to_datetime(na['D']['lab'].TimeStamp))
for product in ['rg3','cn7']:
 for kind,priority in [('labeled',0),('unlabeled',1)]:
  f=data_root/'original_split'/f'moldset_{kind}_{product}.csv'
  raw=pd.read_csv(f,parse_dates=['TimeStamp'])
  raw=raw[raw.EQUIP_CD.eq('S14')].copy()
  raw['product']=product;raw['priority']=priority
  records.append(raw[['TimeStamp','product','priority']+sensor_names])
  if kind=='labeled' and product=='rg3':known_timestamps.update(raw.TimeStamp)
stream=pd.concat(records,ignore_index=True)
keys=['product','TimeStamp']
ranges=stream.groupby(keys)[sensor_names].max()-stream.groupby(keys)[sensor_names].min()
conflict=ranges.gt(1e-4).any(axis=1).rename('sensor_conflict')
stream=stream.sort_values('priority').drop_duplicates(keys).merge(conflict,on=keys,how='left')
known=stream['product'].eq('rg3')&stream.TimeStamp.isin(known_timestamps)
stream.loc[stream.sensor_conflict&~known,sensor_names]=np.nan
amb=stream.groupby('TimeStamp')['product'].transform('nunique').gt(1)
ambiguous_times=set(stream.loc[amb,'TimeStamp'])
stream=stream[~amb].sort_values('TimeStamp').reset_index(drop=True)
gap=stream.TimeStamp.diff().dt.total_seconds()
boundary=gap.isna()|gap.gt(1800)|stream['product'].ne(stream['product'].shift())
stream['segment']=boundary.cumsum()
stream['run_id']='U30_'+stream.segment.astype(str).str.zfill(5)
stream['history_count']=stream.groupby('segment').cumcount()
# A uses current-inclusive std10; C/D dense history excludes the current shot, exactly as training.
for v in sensor_names:
 previous=stream.groupby('segment')[v].shift(1)
 stream[v+'__delta1']=stream[v]-previous
 stream[v+'__past10_std']=previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=5).std())
 med=previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=5).median())
 stream[v+'__dev10']=stream[v]-med
 stream[v+'__current_std10']=stream[v].groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=3).std())
 stream[v+'__sparse_dev']=stream[v]-previous.groupby(stream.segment).transform(lambda s:s.rolling(10,min_periods=3).median())
excluded_overlap=int(target.TimeStamp.isin(known_timestamps).sum())
target=target[~target.TimeStamp.isin(known_timestamps)].copy()
excluded_ambiguous=int(target.TimeStamp.isin(ambiguous_times).sum())
target=target[~target.TimeStamp.isin(ambiguous_times)].copy()
u=stream[stream['product'].eq('rg3')&stream.TimeStamp.isin(target.TimeStamp)].sort_values('TimeStamp').reset_index(drop=True)
assert not u.TimeStamp.isin(known_timestamps).any() and u.TimeStamp.is_unique
nu=len(u)
stdU=u[[v+'__current_std10' for v in sensor_names]].to_numpy(float)
ratio=lambda a,b:a/b.where(b.abs()>1e-6)
physU=np.column_stack([
 u.Max_Injection_Pressure-u.Max_Switch_Over_Pressure,
 ratio(u.Max_Switch_Over_Pressure,u.Max_Injection_Pressure),
 ratio(u.Filling_Time,u.Injection_Time),
 ratio(u.Filling_Time,u.Max_Injection_Speed),
 u.Plasticizing_Position-u.Cushion_Position,
 u.Plasticizing_Position__delta1-u.Cushion_Position__delta1,
 ratio(u.Plasticizing_Time,u.Cycle_Time),
 u.Plasticizing_Time__sparse_dev,
 u.Average_Back_Pressure__sparse_dev,
 u.Barrel_Temperature_5-(u.Mold_Temperature_3+u.Mold_Temperature_4)/2,
 u.Max_Injection_Pressure__sparse_dev,
 u.Cushion_Position__sparse_dev])
shockU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Cushion_Position__past10_std])
jointU=np.column_stack([u.Hopper_Temperature__dev10,u.Barrel_Temperature_4__dev10,u.Barrel_Temperature_5__delta1,u.Barrel_Temperature_3__delta1,u.Mold_Temperature_4__delta1,u.Hopper_Temperature__past10_std,u.Max_Injection_Speed__past10_std,u.Cushion_Position__past10_std,u.Plasticizing_Time__past10_std,u.Barrel_Temperature_5__past10_std,u.Average_Back_Pressure__past10_std])
contextU=u[['Barrel_Temperature_5','Mold_Temperature_3','Hopper_Temperature','Max_Injection_Speed']].to_numpy(float)
for ns in [na,nb,nd]+nc:
 ns['y']=np.r_[yn,np.full(nu,-1)]
 ns['g']=np.r_[gn,u.run_id.to_numpy()]
 ns['memo']={}
na['views']['std24']=np.vstack([na['views']['std24'],stdU])
nb['views']['physical12']=np.vstack([nb['views']['physical12'],physU])
nb['context']=np.vstack([nb['context'],contextU])
for ns in nc:ns['thermal']['shock6']=np.vstack([ns['thermal']['shock6'],shockU])
nd['jointbase']=np.vstack([nd['jointbase'],jointU])
ref=np.flatnonzero(usable)
def scores(ref,q):
 return np.column_stack([na['predict'](ref,q)['std24_geometric'],nb['predict'](ref,q)['localphysical12_mean'],np.mean([ns['predict'](ref,q)['shock6_mean'] for ns in nc],axis=0),nd['predict'](ref,q)])
inner=np.full((len(ref),4),np.nan)
for block in pd.unique(gn[ref]):
 mask=gn[ref]==block
 inner[mask]=scores(ref[~mask],ref[mask])
 print('calibration',block,flush=True)
assert np.isfinite(inner).all()
normal=yn[ref]==0
ca=na['threshold'](yn[ref],inner[:,0],.03);cb=nb['threshold'](yn[ref],inner[:,1],.03)
cc=float(np.quantile(inner[normal,2],.99,method='higher'));cd=float(np.quantile(inner[normal,3],.995,method='higher'))
dense=nc[0]['dense']
bs=dense.Barrel_Temperature_5__past10_std.to_numpy();hp=-dense.Hopper_Temperature__dev10.to_numpy()
bcut=float(np.quantile(bs[ref][normal&np.isfinite(bs[ref])],.95,method='higher'))
hcut=float(np.quantile(hp[ref][normal&np.isfinite(hp[ref])],.95,method='higher'))
q=np.arange(nlab,nlab+nu)
score=scores(ref,q)
A=score[:,0]>=ca;B=score[:,1]>=cb
C=(score[:,2]>=cc)&(u.Barrel_Temperature_5__past10_std.to_numpy()>=bcut)
D=(score[:,3]>=cd)&(-u.Hopper_Temperature__dev10.to_numpy()>=hcut)
u['A']=A.astype(int);u['B']=B.astype(int);u['C']=C.astype(int);u['D']=D.astype(int)
u['raw_prediction']=(A|B|C|D).astype(int)
for j,name in enumerate(['A','B','C','D']):u[name+'_score']=score[:,j]
current_valid=u[sensor_names].notna().all(axis=1)
feature_ready=np.isfinite(stdU).all(axis=1)&np.isfinite(physU).all(axis=1)&np.isfinite(shockU).all(axis=1)&np.isfinite(jointU).all(axis=1)&u.history_count.ge(10).to_numpy()
u['feature_ready']=current_valid&feature_ready&~u.sensor_conflict
# Conservative mode compatibility flag, not a quality verdict or a validated acceptance boundary.
profile_cols=['Injection_Time','Cycle_Time','Cushion_Position','Plasticizing_Position']
bounds={}
mode_match=np.ones(nu,bool)
for v in profile_cols:
 train=lab.loc[usable&(yn==0),v+'_raw'].to_numpy(float)
 med=float(np.median(train));iqr=float(np.quantile(train,.75)-np.quantile(train,.25))
 tol=max(abs(med)*.1,10*iqr,.01)
 bounds[v]={'median':med,'lower':med-tol,'upper':med+tol}
 mode_match&=u[v].between(med-tol,med+tol).to_numpy()
u['mode_match']=mode_match
u['applicable']=u.feature_ready&u.mode_match
u['application_status']=np.select([~u.feature_ready,~u.mode_match,u.raw_prediction.eq(1)],['history_or_sensor_unavailable','different_operating_mode','shortshot_candidate'],default='not_flagged')
u['outside_training_time_window']=~u.TimeStamp.between(lab.TimeStamp.min(),lab.TimeStamp.max())
u['prospective_after_training']=u.TimeStamp>lab.TimeStamp.max()
u['known_label_overlap']=False
u['date']=u.TimeStamp.dt.strftime('%Y-%m-%d')
keycols=['TimeStamp','run_id','date','history_count','sensor_conflict','feature_ready','mode_match','applicable','application_status','raw_prediction','A','B','C','D','A_score','B_score','C_score','D_score','outside_training_time_window','prospective_after_training','known_label_overlap']+profile_cols
u[keycols].to_csv(t/'rg3_shortshot_145_unlabeled_predictions.csv',index=False,encoding='utf-8-sig')
candidate=u[u.applicable&u.raw_prediction.eq(1)][keycols]
candidate.to_csv(t/'rg3_shortshot_145_unlabeled_candidates.csv',index=False,encoding='utf-8-sig')
summary=[]
for scope,mask in [('all_target_RH',np.ones(nu,bool)),('feature_ready',u.feature_ready),('mode_match_and_ready',u.applicable),('different_mode_and_ready',u.feature_ready&~u.mode_match)]:
 z=u[mask]
 summary.append(dict(scope=scope,shots=len(z),flagged=int(z.raw_prediction.sum()),flag_rate=float(z.raw_prediction.mean()) if len(z) else np.nan,A=int(z.A.sum()),B=int(z.B.sum()),C=int(z.C.sum()),D=int(z.D.sum())))
sm=pd.DataFrame(summary)
sm.to_csv(t/'rg3_shortshot_145_unlabeled_summary.csv',index=False,encoding='utf-8-sig')
daily=u.groupby('date').agg(shots=('TimeStamp','size'),feature_ready=('feature_ready','sum'),applicable=('applicable','sum'),raw_flagged=('raw_prediction','sum')).reset_index()
accepted=u[u.applicable].groupby('date').raw_prediction.agg(['size','sum']).rename(columns={'size':'applicable_shots','sum':'applicable_flagged'})
daily=daily.merge(accepted,on='date',how='left').fillna(0)
daily['applicable_flag_rate']=daily.applicable_flagged/daily.applicable_shots.replace(0,np.nan)
daily.to_csv(t/'rg3_shortshot_145_unlabeled_daily.csv',index=False,encoding='utf-8-sig')
runs=u.groupby('run_id').agg(start=('TimeStamp','min'),end=('TimeStamp','max'),shots=('TimeStamp','size'),applicable=('applicable','sum')).reset_index()
perrun=u[u.applicable].groupby('run_id').raw_prediction.agg(['size','sum']).rename(columns={'size':'applicable_shots','sum':'applicable_flagged'})
runs=runs.merge(perrun,on='run_id',how='left').fillna(0)
runs['applicable_flag_rate']=runs.applicable_flagged/runs.applicable_shots.replace(0,np.nan)
runs.to_csv(t/'rg3_shortshot_145_unlabeled_runs.csv',index=False,encoding='utf-8-sig')
shift=[]
trainstd=na['views']['std24'][:nlab][ref[normal]]
for j,name in enumerate(sensor_names):
 vals=stdU[u.applicable,j]
 lo=float(np.nanquantile(trainstd[:,j],.01));hi=float(np.nanquantile(trainstd[:,j],.99))
 shift.append(dict(feature=name+'_std10',train_normal_median=float(np.nanmedian(trainstd[:,j])),unlabeled_applicable_median=float(np.nanmedian(vals)) if len(vals) else np.nan,train_p01=lo,train_p99=hi,outside_train_p01_p99=float(np.mean((vals<lo)|(vals>hi))) if len(vals) else np.nan))
pd.DataFrame(shift).to_csv(t/'rg3_shortshot_145_unlabeled_feature_shift.csv',index=False,encoding='utf-8-sig')
audit=dict(training_normal=int(normal.sum()),training_shortshot=int((yn[ref]==1).sum()),unlabeled_training_rows=0,
 unlabeled_RH_raw_rows=target_raw_count,unlabeled_RH_unique_rows=int(target_rows.TimeStamp.nunique()),
 excluded_known_label_timestamps=excluded_overlap,excluded_ambiguous_timestamps=excluded_ambiguous,
 inference_shots=nu,inference_known_label_overlap=0,history_conflicts=int(u.sensor_conflict.sum()),
 A_threshold=ca,B_threshold=cb,C_threshold=cc,D_threshold=cd,
 C_barrel5_std_threshold=bcut,D_hopper_cooling_threshold=hcut,
 mode_bounds=bounds,history_gap_seconds=1800,history_window=10,
 prospective_after_training=int(u.prospective_after_training.sum()),
 training_start=str(lab.TimeStamp.min()),training_end=str(lab.TimeStamp.max()),
 sparse_to_dense_window_shift=True,metrics_without_labels='F1/TP/FP/precision/recall unavailable')
(t/'rg3_shortshot_145_unlabeled_audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2),encoding='utf-8')
print(sm.to_string(index=False))
print('STATUS',u.application_status.value_counts().to_string())
print('DAILY APPLICABLE',daily[daily.applicable_shots>0].to_string(index=False))
print('AUDIT',json.dumps(audit,ensure_ascii=False))
